# Treino Final ASL - Versão Robusta (Correção de Posição)

## O Problema Anterior
O modelo anterior falhava porque aprendia a **posição absoluta** da mão (ex: "A letra A acontece no pixel 500"). Quando a mão se movia, o modelo ficava confuso.

## A Solução Implementada
Neste notebook, aplicamos o pré-processamento padrão da indústria para reconhecimento de gestos:
1.  **Coordenadas Relativas:** Subtraímos a posição do Pulso (Ponto 0) de todos os outros pontos. A mão passa a ser reconhecida em qualquer lugar do ecrã.
2.  **Remoção de Ruído (Z):** Ignoramos a coordenada Z (profundidade), pois ela é instável em webcams comuns. Usamos apenas X e Y.

**Resultado:** Um modelo que funciona mesmo que a mão esteja no canto do ecrã ou a distâncias diferentes.

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_validate
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

# Algoritmos
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

# Caminhos
DATA_PATH = 'hand_landmarks_dataset.csv'
MODELS_DIR = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)

## 1. Carregamento e Transformação Matemática (O Passo Mágico)

In [ ]:
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Ficheiro {DATA_PATH} não encontrado.")

df = pd.read_csv(DATA_PATH)
print(f"Dataset Original: {df.shape} (com X, Y, Z absolutos)")

X_raw = df.drop(columns=['label', 'hand'], errors='ignore').values
y_labels = df['label'].values

X_processed = []

print("🔄 A converter coordenadas para Relativas e a remover Z...")

for row in X_raw:
    # 1. Recuperar estrutura dos 21 pontos (x, y, z)
    landmarks = row.reshape(21, 3)
    
    # 2. Definir o Pulso (Ponto 0) como a origem (0,0)
    wrist = landmarks[0]
    
    # 3. Subtrair o pulso de todos os pontos
    # (Isto torna o gesto invariante à posição no ecrã)
    relative_landmarks = landmarks - wrist
    
    # 4. REMOVER O Z (Ficar apenas com colunas 0 e 1 -> X e Y)
    # Isto remove o ruído e aumenta a estabilidade
    relative_xy = relative_landmarks[:, :2] 
    
    # 5. Aplanar para vetor de 42 valores (21 pontos * 2 coords)
    X_processed.append(relative_xy.flatten())

X = np.array(X_processed)
print(f"✅ Processamento concluído. Novo formato: {X.shape} (42 features)")

# Label Encoding
le = LabelEncoder()
y = le.fit_transform(y_labels)
print(f"Classes detetadas: {le.classes_}")

## 2. Divisão e Normalização

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# StandardScaler é fundamental para SVM e MLP funcionarem bem
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Treino: {X_train_s.shape}, Teste: {X_test_s.shape}")

## 3. Seleção do Melhor Modelo
Vamos testar rapidamente qual o algoritmo que se porta melhor com estes novos dados.

In [ ]:
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'Neural Network': MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42)
}

best_score = 0
best_model_name = ""
best_model = None

print("🏁 A comparar modelos...")
for name, model in models.items():
    # Cross Validation de 5 folds
    cv_res = cross_validate(model, X_train_s, y_train, cv=5, scoring='accuracy', n_jobs=-1)
    acc = cv_res['test_score'].mean()
    print(f"-> {name}: Acurácia = {acc:.4f}")
    
    if acc > best_score:
        best_score = acc
        best_model_name = name
        best_model = model

print(f"\n🏆 VENCEDOR: {best_model_name} com {best_score:.4f}")

## 4. Treino Final e Avaliação
Treinamos o modelo vencedor com todos os dados de treino.

In [ ]:
print(f"A treinar {best_model_name} final...")
best_model.fit(X_train_s, y_train)

y_pred = best_model.predict(X_test_s)
print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Matriz de Confusão
plt.figure(figsize=(12, 10))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=False, cmap='Blues', 
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f'Matriz de Confusão - {best_model_name}')
plt.show()

## 5. Guardar Artefactos
⚠️ **Importante:** Estes novos ficheiros esperam receber apenas 42 números (X e Y relativos). Tens de atualizar o teu `client_app.py` para fazer a mesma matemática.

In [ ]:
print("💾 A guardar modelos...")
joblib.dump(best_model, os.path.join(MODELS_DIR, 'best_model.pkl'))
joblib.dump(scaler, os.path.join(MODELS_DIR, 'scaler_hand_sign.pkl'))
joblib.dump(le, os.path.join(MODELS_DIR, 'label_encoder.pkl'))
print("✅ Feito! Reinicia o teu app.py para carregar este novo cérebro.")